# Sentinel-2 super-resolution with an ESRI World Imagery guide (hyper-ez + hyEnhance)

This notebook applies the **P2SR** enhancement method from
[hyEnhance](https://github.com/naikp13/hyEnhance) (Naik et al., 2025, *Remote Sensing* 17(11), 1878,
[doi:10.3390/rs17111878](https://doi.org/10.3390/rs17111878)) to **Sentinel-2** instead of a hyperspectral image.

In hyper-ez the method takes a low-resolution spectral image and a high-resolution guide image. Here the roles are:

| Role in `HSIEnhancer` | hyper-ez (hyperspectral workflow) | This notebook |
|---|---|---|
| low-resolution spectral image (`hsi_path`) | EnMAP 30 m (224 bands) | **Sentinel-2 L2A**, 12 bands, 10 m grid (20 m / 60 m bands resampled to 10 m) |
| high-resolution guide (`msi_path`) | Sentinel-2 10 m, then ESRI 2 m | **ESRI World Imagery** RGB, 2 m (configurable) |

The result is a 12-band Sentinel-2 cube on the ESRI grid (10 m to 2 m is a scale factor of 5).

**Workflow**

1. Set the area of interest and the target date.
2. Search Sentinel-2 L2A on the AWS Earth Search STAC API and pick the scene closest to the date that fully covers the area.
3. Read only the needed window of the 12 Sentinel-2 bands (cloud-optimized GeoTIFFs), convert them to surface reflectance and stack them on a 10 m grid.
4. Download ESRI World Imagery for the same footprint at 2 m (hyper-ez `HyperspectralProcessor.acquire_high_resolution_imagery`).
5. Enhance with hyEnhance `HSIEnhancer.fuse_to_enhance` (patch-wise sparse residual, high-frequency detail injection and guided filtering).
6. Save a georeferenced GeoTIFF and compare the original, enhanced and ESRI images.
7. Check spectral consistency: aggregate the 2 m result back to 10 m and compare it with the original bands.
8. Optional reduced-resolution check (Wald protocol) that needs no external reference: degrade Sentinel-2 to 50 m, enhance it back to 10 m with its own 10 m RGB as the guide, and score the result against the real 10 m bands.

The notebook runs on Google Colab or locally (Python 3.10+). It needs internet access to GitHub, PyPI, `earth-search.aws.element84.com`, `sentinel-cogs.s3.us-west-2.amazonaws.com` and `services.arcgisonline.com`.

## 1. Setup

In [ ]:
# Clone hyper-ez and hyEnhance next to this notebook (skipped if they already exist)
# and install any missing dependencies.
import importlib.util
import os
import subprocess
import sys

REPOS = {
    "hyper-ez": "https://github.com/naikp13/hyper-ez.git",
    "hyEnhance": "https://github.com/naikp13/hyEnhance.git",  # must keep this folder name: hyper-ez imports `hyEnhance.src`
}
REPO_ROOT = os.environ.get("HYPEREZ_REPO_ROOT", os.getcwd())
for name, url in REPOS.items():
    path = os.path.join(REPO_ROOT, name)
    if not os.path.isdir(path):
        subprocess.run(["git", "clone", "--depth", "1", url, path], check=True)

for p in (os.path.join(REPO_ROOT, "hyper-ez"), REPO_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

# module name -> pip package
REQUIRED = {
    "numpy": "numpy", "scipy": "scipy", "rasterio": "rasterio", "shapely": "shapely",
    "pystac_client": "pystac-client", "requests": "requests", "pytz": "pytz",
    "matplotlib": "matplotlib", "tqdm": "tqdm", "joblib": "joblib", "numba": "numba",
    "pywt": "PyWavelets", "sklearn": "scikit-learn", "skimage": "scikit-image",
}
def has_cv2_contrib():
    # Probe in a separate process so this kernel never caches a cv2 without `ximgproc`
    # (the guided filter lives in opencv-contrib)
    probe = "import cv2, sys; sys.exit(0 if hasattr(cv2, 'ximgproc') else 1)"
    return subprocess.run([sys.executable, "-c", probe], capture_output=True).returncode == 0


missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if not has_cv2_contrib():
    missing.append("opencv-contrib-python")
if missing:
    print("Installing:", " ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
    if "opencv-contrib-python" in missing and not has_cv2_contrib():
        raise RuntimeError("opencv-contrib-python is installed but cv2 still has no `ximgproc`. Uninstall "
                           "opencv-python / opencv-python-headless, reinstall opencv-contrib-python, then restart the kernel.")
    if "cv2" in sys.modules:
        raise RuntimeError("OpenCV was replaced after it was imported. Restart the kernel and run the notebook again.")
else:
    print("All dependencies are installed.")

In [ ]:
import logging
import math
import shutil
import warnings
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pytz
import rasterio
from pystac_client import Client
from rasterio.coords import BoundingBox
from rasterio.crs import CRS
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.warp import Resampling, reproject, transform as transform_coords
from rasterio.windows import from_bounds as window_from_bounds
from shapely.geometry import box, shape
from skimage.transform import resize

# hyper-ez: configuration and ESRI World Imagery download
from hyper_ez.config import STAC_API_URL, SENTINEL2_BAND_MAPPING, setup_matplotlib_style
from hyper_ez.image_processing import HyperspectralProcessor

# hyEnhance: the P2SR enhancement method
from hyEnhance.src import HSIEnhancer, HSIDataLoader, PatchProcessor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
setup_matplotlib_style()
plt.rcParams["font.size"] = 12

## 2. Parameters

The default area is the Marienkas validation site used in hyper-ez (southern Namibia), 3 km × 3 km.
Change `AOI_CENTER_LON`, `AOI_CENTER_LAT`, `AOI_SIZE_M` and `TARGET_DATE` for your own site.

- Keep the area small on free Colab: 3 km × 3 km gives a 300 × 300 Sentinel-2 cube and a 1500 × 1500 ESRI image.
  The ESRI export service returns at most 4096 pixels per side.
- `ESRI_RES_M` must divide 10 m evenly (5 m, 2.5 m, 2 m or 1 m); the scale factor is `10 / ESRI_RES_M`.
- The enhancement parameters are those of `HSIEnhancer.fuse_to_enhance`. hyper-ez uses `GUIDE_RADIUS = 10` and
  `DETAIL_WEIGHT = 1` for hyperspectral data. For Sentinel-2, `GUIDE_RADIUS = 4` and `DETAIL_WEIGHT = 0.25` scored best in
  the reduced-resolution check of section 9 on the default site (it beat bicubic on RMSE, ERGAS and correlation, while
  `DETAIL_WEIGHT = 1` did not). Re-run section 9 on your own site to confirm.
- `STRIDE = 2` processes a quarter of the patches of `STRIDE = 1` (the hyper-ez default).

In [ ]:
# Area of interest (WGS84) and date
AOI_CENTER_LON = 17.40713
AOI_CENTER_LAT = -28.13193
AOI_SIZE_M = 3000            # side length of the square area, in metres
TARGET_DATE = "2024-07-20"   # YYYY-MM-DD; the closest cloud-free scene is used
SEARCH_WINDOW_DAYS = 30
MAX_CLOUD_COVER = 5          # percent

# Grids
S2_RES_M = 10.0
ESRI_RES_M = 2.0

# P2SR / HSIEnhancer parameters
PATCH_SIZE = 12
STRIDE = 2
GUIDE_RADIUS = 4      # hyper-ez default: 10
DETAIL_WEIGHT = 0.25  # hyper-ez default: 1

# Sentinel-2 bands in hyper-ez order (B01..B12, B8A after B08)
S2_BANDS = list(SENTINEL2_BAND_MAPPING.keys())
OUT_DIR = "s2_esri_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

SCALE = S2_RES_M / ESRI_RES_M
assert abs(SCALE - round(SCALE)) < 1e-9, "ESRI_RES_M must divide 10 m evenly"
SCALE = int(round(SCALE))
print(f"Scale factor: {SCALE}x  ({S2_RES_M:g} m -> {ESRI_RES_M:g} m)")

## 3. Find a Sentinel-2 L2A scene

In [ ]:
def aoi_lonlat_box(lon, lat, size_m):
    # A WGS84 box slightly larger than the square AOI, used for the STAC search
    half_deg_lat = (size_m / 2) / 111_320 * 1.05
    half_deg_lon = half_deg_lat / max(math.cos(math.radians(lat)), 1e-6)
    return box(lon - half_deg_lon, lat - half_deg_lat, lon + half_deg_lon, lat + half_deg_lat)


def search_sentinel2(aoi, target_date, days=SEARCH_WINDOW_DAYS, max_cloud=MAX_CLOUD_COVER):
    # Closest-in-time L2A scene whose footprint fully contains the AOI
    t0 = datetime.strptime(target_date, "%Y-%m-%d").replace(tzinfo=pytz.UTC)
    rng = f"{(t0 - timedelta(days=days)):%Y-%m-%d}/{(t0 + timedelta(days=days)):%Y-%m-%d}"
    items = list(Client.open(STAC_API_URL).search(
        collections=["sentinel-2-l2a"], intersects=aoi.__geo_interface__,
        datetime=rng, query={"eo:cloud_cover": {"lt": max_cloud}},
    ).items())
    covering = [it for it in items if shape(it.geometry).contains(aoi)]
    if not covering:
        raise RuntimeError(f"No scene with <{max_cloud}% cloud fully covers the AOI in {rng}. "
                           "Widen SEARCH_WINDOW_DAYS, raise MAX_CLOUD_COVER or move the AOI.")
    return min(covering, key=lambda it: (abs(it.datetime - t0), it.properties["eo:cloud_cover"]))


aoi_wgs84 = aoi_lonlat_box(AOI_CENTER_LON, AOI_CENTER_LAT, AOI_SIZE_M)
s2_item = search_sentinel2(aoi_wgs84, TARGET_DATE)
print(f"Scene: {s2_item.id}")
print(f"Date: {s2_item.datetime:%Y-%m-%d}   cloud cover: {s2_item.properties['eo:cloud_cover']:.2f}%")

## 4. Build the working grid and read the Sentinel-2 bands

The grid uses the scene's own UTM zone. Its corners are snapped to multiples of 60 m, so the 10, 20 and 60 m Sentinel-2
pixels and the 2 m ESRI pixels all nest exactly. Only the needed window of each cloud-optimized GeoTIFF is read
(a few MB rather than the full 110 km tile). Digital numbers are converted to surface reflectance with the scale and
offset in the STAC metadata. Scenes from processing baseline 04.00 onward carry a +1000 offset; it is removed here unless
Earth Search has already removed it (`earthsearch:boa_offset_applied`).

In [ ]:
def scene_crs(item):
    code = item.properties.get("proj:code") or f"EPSG:{item.properties['proj:epsg']}"
    return CRS.from_string(code)


def snapped_bounds(lon, lat, size_m, crs, snap=60.0):
    xs, ys = transform_coords(CRS.from_epsg(4326), crs, [lon], [lat])
    n = math.ceil(size_m / snap)
    left = math.floor((xs[0] - n * snap / 2) / snap) * snap
    bottom = math.floor((ys[0] - n * snap / 2) / snap) * snap
    return BoundingBox(left, bottom, left + n * snap, bottom + n * snap)


def band_scale_offset(asset, item):
    # Earth Search has already removed the BOA offset from many scenes and flags them with
    # `earthsearch:boa_offset_applied`; applying the -0.1 offset again would make dark pixels negative.
    rb = (asset.extra_fields.get("raster:bands") or [{}])[0]
    scale = rb.get("scale", 1e-4)
    if item.properties.get("earthsearch:boa_offset_applied", False):
        return scale, 0.0
    if "offset" in rb:
        return scale, rb["offset"]
    baseline = float(item.properties.get("s2:processing_baseline", "0") or 0)
    return scale, (-0.1 if baseline >= 4.0 else 0.0)


def read_s2_band(item, band, bounds, crs, res):
    asset = item.assets[SENTINEL2_BAND_MAPPING[band]]
    scale, offset = band_scale_offset(asset, item)
    width = int(round((bounds.right - bounds.left) / res))
    height = int(round((bounds.top - bounds.bottom) / res))
    dst_transform = transform_from_bounds(*bounds, width, height)
    out = np.full((height, width), np.nan, dtype=np.float32)
    env = dict(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", AWS_NO_SIGN_REQUEST="YES",
               CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif")
    with rasterio.Env(**env), rasterio.open(asset.href) as src:
        pad = 2 * max(src.res)  # margin for bilinear resampling of 20/60 m bands
        win = window_from_bounds(bounds.left - pad, bounds.bottom - pad, bounds.right + pad,
                                 bounds.top + pad, transform=src.transform)
        win = win.round_offsets().round_lengths()
        dn = src.read(1, window=win).astype(np.float32)
        refl = np.where(dn == 0, np.nan, dn * scale + offset).astype(np.float32)
        reproject(refl, out, src_transform=src.window_transform(win), src_crs=src.crs,
                  src_nodata=np.nan, dst_transform=dst_transform, dst_crs=crs,
                  dst_nodata=np.nan, resampling=Resampling.bilinear)
    return out


def write_geotiff(path, data_hwc, bounds, crs, band_names=None, dtype="float32"):
    h, w, c = data_hwc.shape
    profile = dict(driver="GTiff", height=h, width=w, count=c, dtype=dtype, crs=crs,
                   transform=transform_from_bounds(*bounds, w, h), compress="deflate",
                   tiled=True, blockxsize=256, blockysize=256)
    if dtype == "float32":
        profile["nodata"] = np.nan
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(np.moveaxis(data_hwc, -1, 0).astype(dtype))
        for i, name in enumerate(band_names or [], start=1):
            dst.set_band_description(i, name)


work_crs = scene_crs(s2_item)
work_bounds = snapped_bounds(AOI_CENTER_LON, AOI_CENTER_LAT, AOI_SIZE_M, work_crs)
print(f"CRS: {work_crs}   bounds: {tuple(work_bounds)}")

s2 = np.stack([read_s2_band(s2_item, b, work_bounds, work_crs, S2_RES_M) for b in S2_BANDS], axis=-1)
s2_path = os.path.join(OUT_DIR, "sentinel2_10m.tif")
write_geotiff(s2_path, s2, work_bounds, work_crs, S2_BANDS)
print(f"Sentinel-2 stack: {s2.shape}  (rows, cols, bands) -> {s2_path}")
print(f"Reflectance range: {np.nanmin(s2):.3f} .. {np.nanmax(s2):.3f}   no-data pixels: {np.isnan(s2).any(-1).sum()}")

## 5. Download the ESRI World Imagery guide

This uses hyper-ez's `HyperspectralProcessor.acquire_high_resolution_imagery`, which requests the ESRI export service
in Web Mercator and reprojects the image onto the working grid at `ESRI_RES_M`.

In [ ]:
esri_w = int(round((work_bounds.right - work_bounds.left) / ESRI_RES_M))
esri_h = int(round((work_bounds.top - work_bounds.bottom) / ESRI_RES_M))
assert max(esri_w, esri_h) <= 4096, "ESRI export is limited to 4096 px per side: shrink AOI_SIZE_M or raise ESRI_RES_M"

hsi_proc = HyperspectralProcessor()
downloaded = hsi_proc.acquire_high_resolution_imagery(work_bounds, work_crs, target_resolution=ESRI_RES_M)

with rasterio.open(downloaded) as src:
    esri = np.moveaxis(src.read(), 0, -1)[..., :3]  # keep R, G, B
esri_path = os.path.join(OUT_DIR, f"esri_{ESRI_RES_M:g}m.tif")
write_geotiff(esri_path, esri, work_bounds, work_crs, ["Red", "Green", "Blue"], dtype="uint8")
os.remove(downloaded)

assert esri.shape[0] == SCALE * s2.shape[0] and esri.shape[1] == SCALE * s2.shape[1], \
    f"Grid mismatch: ESRI {esri.shape[:2]} vs {SCALE} x Sentinel-2 {s2.shape[:2]}"
print(f"ESRI guide: {esri.shape} -> {esri_path}")

In [ ]:
def stretch(rgb, ref=None, lo=2, hi=98):
    # Percentile stretch to 0..1; pass `ref` to reuse another image's limits
    ref = rgb if ref is None else ref
    p_lo = np.nanpercentile(ref, lo, axis=(0, 1))
    p_hi = np.nanpercentile(ref, hi, axis=(0, 1))
    return np.clip((rgb - p_lo) / (p_hi - p_lo + 1e-12), 0, 1)


def bands(cube, names, band_list):
    return cube[..., [band_list.index(n) for n in names]]


TRUE_COLOR = ["B04", "B03", "B02"]
FALSE_COLOR = ["B08", "B04", "B03"]

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
axes[0].imshow(stretch(bands(s2, TRUE_COLOR, S2_BANDS)), interpolation="nearest")
axes[0].set_title(f"Sentinel-2 true colour, {S2_RES_M:g} m")
axes[1].imshow(esri)
axes[1].set_title(f"ESRI World Imagery, {ESRI_RES_M:g} m")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## 6. Enhance Sentinel-2 with the P2SR method

`HSIEnhancer(msi_path, hsi_path)` loads both images, drops spectral bands with more than 50 % no-data and fills small
gaps. `fuse_to_enhance` then

1. runs the patch-wise sparse residual step (`PatchProcessor.run_parallel`): wavelet, FastICA and NMF components of each
   Sentinel-2 patch and a dictionary learned with the degraded guide are used to predict high-resolution residuals;
2. upsamples each band to the guide grid and injects the guide's high-frequency detail, then restores each band's
   original mean and standard deviation so the radiometry is kept (`HSIUpsampler`);
3. applies a guided filter per band with the ESRI image as guide (`GUIDE_RADIUS`).

The run takes a few minutes for the default 300 × 300 area.

In [ ]:
enhancer = HSIEnhancer(esri_path, s2_path)
kept = np.isnan(HSIDataLoader.load_image(s2_path)).mean(axis=(0, 1)) <= 0.5
kept_bands = [b for b, k in zip(S2_BANDS, kept) if k]
if len(kept_bands) < len(S2_BANDS):
    print("Dropped (mostly no-data):", [b for b in S2_BANDS if b not in kept_bands])

s2_enhanced = enhancer.fuse_to_enhance(
    patch_size=PATCH_SIZE, stride=STRIDE, guide_radius=GUIDE_RADIUS, detail_weight=DETAIL_WEIGHT
).astype(np.float32)

enhanced_path = os.path.join(OUT_DIR, f"sentinel2_enhanced_{ESRI_RES_M:g}m.tif")
write_geotiff(enhanced_path, s2_enhanced, work_bounds, work_crs, kept_bands)
print(f"Enhanced Sentinel-2: {s2_enhanced.shape} -> {enhanced_path}")

**Diagnostic: is the patch-wise residual step contributing?** `PatchProcessor.process_patch` catches any error, logs it
to `processing.log` and skips the patch, so a failing step does not stop the run. This cell runs a few patches in the
notebook process and prints the first error, so you can see whether step 1 above is active with your installed library
versions. If every patch fails, the result comes from steps 2 and 3 only.

In [ ]:
class _Collect(logging.Handler):
    def __init__(self):
        super().__init__(); self.messages = []
    def emit(self, record):
        self.messages.append(record.getMessage())

probe = PatchProcessor(enhancer.hsi, enhancer.msi, enhancer.n_components, enhancer.n_atoms, enhancer.lambda_reg)
probe.f = SCALE
probe.msi_lr = resize(enhancer.msi, (enhancer.hsi.shape[0], enhancer.hsi.shape[1], enhancer.msi.shape[2]),
                      order=2, preserve_range=True)
collector = _Collect(); logging.getLogger().addHandler(collector)
step = max(1, (enhancer.hsi.shape[0] - PATCH_SIZE) // 4)
coords = [(x, x) for x in range(0, enhancer.hsi.shape[0] - PATCH_SIZE + 1, step)][:5]
ok = sum(probe.process_patch(x, y, PATCH_SIZE) is not None for x, y in coords)
logging.getLogger().removeHandler(collector)

print(f"{ok} of {len(coords)} probe patches produced a sparse residual.")
if ok < len(coords) and collector.messages:
    print("First error:", collector.messages[0])

## 7. Visual comparison

In [ ]:
def compare(composite, title, zoom=None):
    lr = bands(s2, composite, S2_BANDS)
    hr = bands(s2_enhanced, composite, kept_bands)
    lr_up = np.repeat(np.repeat(lr, SCALE, 0), SCALE, 1)  # nearest neighbour, to show the 10 m pixels
    panels = [(stretch(lr_up, lr), f"Sentinel-2 {S2_RES_M:g} m"),
              (stretch(hr, lr), f"Enhanced {ESRI_RES_M:g} m"),
              (esri / 255.0, f"ESRI guide {ESRI_RES_M:g} m")]
    if zoom is not None:
        r0, r1, c0, c1 = zoom
        panels = [(img[r0:r1, c0:c1], t) for img, t in panels]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6.3))
    for ax, (img, t) in zip(axes, panels):
        ax.imshow(img, interpolation="nearest"); ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title, fontweight="bold"); plt.tight_layout(); plt.show()


H, W = s2_enhanced.shape[:2]
zoom = (H // 2 - H // 8, H // 2 + H // 8, W // 2 - W // 8, W // 2 + W // 8)  # central quarter
compare(TRUE_COLOR, "True colour (B04, B03, B02)")
compare(TRUE_COLOR, "True colour, central zoom", zoom)
compare(FALSE_COLOR, "False colour (B08, B04, B03), central zoom", zoom)

## 8. Spectral consistency check

Averaging the enhanced image over each 5 × 5 block should give back the original 10 m Sentinel-2 values. Large
differences mean the enhancement changed the spectra, not only the spatial detail.

Metrics: per-band correlation (CC) and RMSE in reflectance, mean spectral angle (SAM, degrees, 0 is identical) and
ERGAS (dimensionless, lower is better; values under about 3 are usually considered good).

In [ ]:
def block_mean(cube, f):
    h, w, c = cube.shape
    return cube[: h // f * f, : w // f * f].reshape(h // f, f, w // f, f, c).mean(axis=(1, 3))


def quality_metrics(ref, est, ratio):
    # ref, est: (rows, cols, bands) on the same grid; ratio = low-res / high-res pixel size
    valid = np.isfinite(ref).all(-1) & np.isfinite(est).all(-1)
    r, e = ref[valid].astype(np.float64), est[valid].astype(np.float64)
    rmse_b = np.sqrt(((r - e) ** 2).mean(0))
    cc_b = np.array([np.corrcoef(r[:, i], e[:, i])[0, 1] for i in range(r.shape[1])])
    cos = (r * e).sum(1) / (np.linalg.norm(r, axis=1) * np.linalg.norm(e, axis=1) + 1e-12)
    sam = np.degrees(np.arccos(np.clip(cos, -1, 1))).mean()
    ergas = 100.0 / ratio * np.sqrt(np.mean((rmse_b / (np.abs(r.mean(0)) + 1e-12)) ** 2))
    psnr = 20 * np.log10(r.max() / (np.sqrt(((r - e) ** 2).mean()) + 1e-12))
    return dict(RMSE=np.sqrt(((r - e) ** 2).mean()), PSNR=psnr, SAM=sam, ERGAS=ergas, CC=cc_b.mean(),
                rmse_band=rmse_b, cc_band=cc_b)


s2_kept = bands(s2, kept_bands, S2_BANDS)
back = block_mean(s2_enhanced, SCALE)
m = quality_metrics(s2_kept, back, SCALE)
print(f"Enhanced -> {S2_RES_M:g} m vs original:  RMSE {m['RMSE']:.4f}   SAM {m['SAM']:.2f} deg   "
      f"ERGAS {m['ERGAS']:.2f}   mean CC {m['CC']:.3f}")

BLUE, ORANGE = "#2a78d6", "#eb6834"
x = np.arange(len(kept_bands))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
axes[0].plot(x, np.nanmean(s2_kept, axis=(0, 1)), "-o", color=BLUE, lw=2, ms=8, label=f"Sentinel-2 {S2_RES_M:g} m")
axes[0].plot(x, np.nanmean(s2_enhanced, axis=(0, 1)), "--s", color=ORANGE, lw=2, ms=8, label=f"Enhanced {ESRI_RES_M:g} m")
axes[0].set_ylabel("Mean surface reflectance"); axes[0].set_title("Scene-mean spectrum"); axes[0].legend(frameon=False)
axes[1].bar(x, m["cc_band"], color=BLUE, width=0.6)
axes[1].set_ylim(min(0.5, m["cc_band"].min() - 0.05), 1.0)
axes[1].set_ylabel("Correlation"); axes[1].set_title(f"Per-band CC, enhanced aggregated to {S2_RES_M:g} m vs original")
for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(kept_bands, rotation=45); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Optional: reduced-resolution check (Wald protocol)

There is no real 2 m Sentinel-2 image to compare with, so this section tests the method at a scale where the truth is
known:

1. degrade the 10 m Sentinel-2 cube to 50 m (5 × 5 block mean, the same factor as 10 m to 2 m);
2. build a 10 m RGB guide from the real Sentinel-2 B04, B03 and B02, stretched to 8-bit like the ESRI image;
3. enhance the 50 m cube back to 10 m with `HSIEnhancer` and compare it with the real 10 m bands, next to a plain bicubic upsampling.

Set `RUN_WALD_CHECK = False` to skip it.

In [ ]:
RUN_WALD_CHECK = True

if RUN_WALD_CHECK:
    f = SCALE
    h, w = (s2.shape[0] // f) * f, (s2.shape[1] // f) * f
    truth = s2_kept[:h, :w]
    low = block_mean(truth, f)
    low_bounds = BoundingBox(work_bounds.left, work_bounds.top - h * S2_RES_M,
                             work_bounds.left + w * S2_RES_M, work_bounds.top)
    guide = (stretch(bands(s2[:h, :w], TRUE_COLOR, S2_BANDS)) * 254 + 1).astype(np.uint8)

    wald_low = os.path.join(OUT_DIR, "wald_s2_low.tif")
    wald_guide = os.path.join(OUT_DIR, "wald_guide_rgb.tif")
    write_geotiff(wald_low, low, low_bounds, work_crs, kept_bands)
    write_geotiff(wald_guide, guide, low_bounds, work_crs, ["Red", "Green", "Blue"], dtype="uint8")

    wald_enh = HSIEnhancer(wald_guide, wald_low).fuse_to_enhance(
        patch_size=PATCH_SIZE, stride=STRIDE, guide_radius=GUIDE_RADIUS, detail_weight=DETAIL_WEIGHT)
    bicubic = resize(low, truth.shape, order=3, mode="edge", preserve_range=True).astype(np.float32)

    results = {"Bicubic": quality_metrics(truth, bicubic, f), "P2SR (hyEnhance)": quality_metrics(truth, wald_enh, f)}
    print(f"{'Method':<18}{'RMSE':>9}{'PSNR dB':>9}{'SAM deg':>9}{'ERGAS':>8}{'CC':>8}")
    for name, r in results.items():
        print(f"{name:<18}{r['RMSE']:>9.4f}{r['PSNR']:>9.2f}{r['SAM']:>9.2f}{r['ERGAS']:>8.2f}{r['CC']:>8.3f}")

    zr, zc = slice(h // 2 - h // 6, h // 2 + h // 6), slice(w // 2 - w // 6, w // 2 + w // 6)
    lim = bands(truth, TRUE_COLOR, kept_bands)
    show = [(np.repeat(np.repeat(low, f, 0), f, 1), "Input 50 m"), (bicubic, "Bicubic 10 m"),
            (wald_enh, "P2SR 10 m"), (truth, "Truth 10 m")]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.4))
    for ax, (img, t) in zip(axes, show):
        ax.imshow(stretch(bands(img, TRUE_COLOR, kept_bands), lim)[zr, zc], interpolation="nearest")
        ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

## 10. Outputs and tuning

Files in `s2_esri_outputs/`:

| File | Content |
|---|---|
| `sentinel2_10m.tif` | 12-band Sentinel-2 L2A surface reflectance on the 10 m working grid |
| `esri_2m.tif` | ESRI World Imagery RGB guide on the same footprint |
| `sentinel2_enhanced_2m.tif` | enhanced Sentinel-2, float32 reflectance, band names in the band descriptions |
| `wald_*.tif` | inputs of the optional reduced-resolution check |

**Tuning**

- `DETAIL_WEIGHT` controls how much ESRI high-frequency detail is injected. Raise it for sharper edges, lower it if
  textures from the RGB image (shadows, roofs) leak into the SWIR or red-edge bands. Watch SAM and ERGAS in section 8.
- `GUIDE_RADIUS` is the guided-filter window in guide pixels. Small values (1 to 3) keep fine detail; larger values
  (around 10) give smoother, more spectrally faithful results.
- `PATCH_SIZE` and `STRIDE` control the patch-wise residual step; `STRIDE = 1` is slowest and matches hyper-ez.

**Caveats**

- ESRI World Imagery is a mosaic acquired on a different date and sensor; land-cover changes between the two dates
  (crops, water, construction) appear as detail in the enhanced bands that Sentinel-2 never saw.
- The 20 m and 60 m bands (B01, B05 to B07, B8A, B09, B11, B12) were resampled to 10 m before enhancement. Their
  real information content stays coarser than the 10 m bands.
- Check the ESRI imagery terms of use before redistributing derived products.

**Citation**: Naik, P., Chakraborty, R., Thiele, S., & Gloaguen, R. (2025). Scalable Hyperspectral Enhancement via
Patch-Wise Sparse Residual Learning: Insights from Super-Resolved EnMAP Data. *Remote Sensing*, 17(11), 1878.
https://doi.org/10.3390/rs17111878